### Data Ingestion pipeline


In [6]:
from langchain_core.documents import Document

document = Document(
            page_content="Hello, world!", metadata={"source": "https://example.com"}
        )

In [1]:
##creat the simple txt file
import os
os.makedirs("../data/textfiles",exist_ok=True)

In [2]:
## TextLoader
from langchain_community.document_loaders import TextLoader

# Initialize the loader with your file path
loader = TextLoader("../data/textfiles/sample1.txt")

# Load the file into a LangChain Document structure
docs = loader.load()

# Accessing the content and metadata
print(docs[0].page_content)  # The actual text content
print(docs[0].metadata)      # Example output: {'source': 'sample_data.txt'}


C:\Users\parth\AppData\Local\Temp\ipykernel_5900\4131449348.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\MYPROJECTS\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample Document 1

This is a sample text file for testing.
It contains a few lines of plain text.
You can use it for RAG ingestion experiments.

{'source': '../data/textfiles/sample1.txt'}


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Loads all .txt files inside a target directory using TextLoader logic
dir_loader = DirectoryLoader(
    path="../data/textfiles",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

all_documents = dir_loader.load()
print(all_documents)


[Document(metadata={'source': '..\\data\\textfiles\\sample1.txt'}, page_content='Sample Document 1\n\nThis is a sample text file for testing.\nIt contains a few lines of plain text.\nYou can use it for RAG ingestion experiments.\n'), Document(metadata={'source': '..\\data\\textfiles\\sample2.txt'}, page_content='Sample Document 2\n\n\nDeep Agents overview\n\n\n\nThe easiest way to start building agents and applications powered by LLMs—with built-in capabilities for task planning, file systems for context management, subagent-spawning, and long-term memory. You can use deep agents for any task, including complex, multi-step tasks.\nDeep Agents is an “agent harness”. It is the same core tool calling loop as other agent frameworks, but with built-in capabilities that make agents reliable for real tasks:\nTake actions in an environment\nTake actions via tools, read and write files, execute code\nConnect to your data\nLoad memories, skills, and domain knowledge at the right moment\nManage g

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader
loader = PyPDFLoader("../data/pdfs/Parth_Dudhe_Amazon_MLSS_SOP.pdf")

# Load pages as an array of Documents
pages = loader.load()

# Access data from the first page
print(pages[0].page_content)
print(pages[0].metadata) # Outputs: {'source': 'example.pdf', 'page': 0}


Statement of Purpose
Amazon Machine Learning Summer School
As a Computer Science undergraduate at IIIT Jabalpur, my technical journey has been driven
by a fascination with how robust engineering and intelligent data processing converge to solve
complex  problems.  I  began  by  building  a  strong  foundation  in  distributed  systems  and
software  engineering,  actively  contributing  to  open-source  protocols  like  Braidpool.  This
experience taught me the rigor required to design scalable architectures, but my curiosity
naturally  gravitated  toward  how  we  can  extract  meaningful  intelligence  from  massive,
unstructured data environments.
To pursue this, I focused heavily on data science fundamentals, studying statistical analysis,
probability distributions, and regression models. This foundational knowledge culminated in
Genesis_KB, an automated Retrieval-Augmented Generation (RAG) pipeline I architected.
The project’s goal was to ingest dense, unstructured technical conte

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader
#fast , used for large files
loader = PyMuPDFLoader("../data/pdfs/Parth_Dudhe_Amazon_MLSS_SOP.pdf")
docs = loader.load()
print(docs)

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': '', 'creationdate': '', 'source': '../data/pdfs/Parth_Dudhe_Amazon_MLSS_SOP.pdf', 'file_path': '../data/pdfs/Parth_Dudhe_Amazon_MLSS_SOP.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': 'Statement of Purpose - Amazon ML Summer School', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Statement of Purpose\nAmazon Machine Learning Summer School\nAs a Computer Science undergraduate at IIIT Jabalpur, my technical journey has been driven\nby a fascination with how robust engineering and intelligent data processing converge to solve\ncomplex problems. I began by building a strong foundation in distributed systems and\nsoftware engineering, actively contributing to open-source protocols like Braidpool. This\nexperience taught me the rigor required to design scalable architectures, but my curiosity\nnaturally  gravitated  toward  how  we  can

## Embedding and VectorDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:


class EmbeddingManager:
    def __init__(self,
                 model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None

        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")

            self.model = SentenceTransformer(
                self.model_name
            )

            print(
                f"Loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_embedding_dimension()}"
            )

        except Exception as e:
            print(e)
            raise

    def generate_embeddings(
        self,
        texts: List[str]
    ) -> np.ndarray:

        if self.model is None:
            raise ValueError("Model not loaded")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        return embeddings

VectorStore


In [8]:
# Prepare documents to store
documents = globals().get("all_documents") or globals().get("docs")
if documents is None:
    if "document" in globals():
        documents = [document]
    else:
        raise NameError("No documents found. Define all_documents, docs, or document first.")

# Create embeddings
embedding_manager = EmbeddingManager()
texts = [doc.page_content for doc in documents]
embeddings = embedding_manager.generate_embeddings(texts).tolist()

# Create a persistent Chroma DB
client = chromadb.PersistentClient(path="../data/chroma_db")
collection = client.get_or_create_collection(name="text_documents")

# Store documents in Chroma
collection.add(
    ids=[str(uuid.uuid4()) for _ in documents],
    documents=texts,
    metadatas=[doc.metadata for doc in documents],
    embeddings=embeddings
)

print(f"Stored {collection.count()} documents in ChromaDB.")

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2916.62it/s]


Loaded successfully. Embedding dimension: 384


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]


Stored 4 documents in ChromaDB.


Retriever pipeline 

In [9]:
# Define query text
query_text = "What is this document about?"

# Reuse existing embedding manager if available
embedding_manager = globals().get("embedding_manager") or EmbeddingManager()

# Generate query embedding
query_embedding = embedding_manager.generate_embeddings([query_text])[0]

print("Query embedding shape:", query_embedding.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00, 50.00it/s]

Query embedding shape: (384,)


In [10]:
# Retrieve top k similar documents from ChromaDB
k = 3  # Number of top results to retrieve

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=k
)

# Display results
print(f"Retrieved top {k} documents:\n")
for i, (doc_id, document, metadata, distance) in enumerate(zip(
    results['ids'][0],
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"Result {i+1}:")
    print(f"  ID: {doc_id}")
    print(f"  Distance: {distance:.4f}")
    print(f"  Metadata: {metadata}")
    print(f"  Content: {document[:200]}...\n")

Retrieved top 3 documents:

Result 1:
  ID: 31485f44-efe7-4a1b-a092-422388bf54db
  Distance: 1.2280
  Metadata: {'source': '..\\data\\textfiles\\sample1.txt'}
  Content: Sample Document 1

This is a sample text file for testing.
It contains a few lines of plain text.
You can use it for RAG ingestion experiments.
...

Result 2:
  ID: b5a13553-fed1-40d3-b42f-a8c31fe9bd29
  Distance: 1.2280
  Metadata: {'source': '..\\data\\textfiles\\sample1.txt'}
  Content: Sample Document 1

This is a sample text file for testing.
It contains a few lines of plain text.
You can use it for RAG ingestion experiments.
...

Result 3:
  ID: a7faf9e8-2dbe-4c58-9eb2-169eebf3d655
  Distance: 1.7061
  Metadata: {'source': '..\\data\\textfiles\\sample2.txt'}
  Content: Sample Document 2


Deep Agents overview



The easiest way to start building agents and applications powered by LLMs—with built-in capabilities for task planning, file systems for context management,...



In [11]:
# Extract context documents from retrieval results
context_docs = []

for i, (doc_id, doc_content, metadata) in enumerate(zip(
    results['ids'][0],
    results['documents'][0],
    results['metadatas'][0]
)):
    context_docs.append({
        'id': doc_id,
        'content': doc_content,
        'metadata': metadata,
        'rank': i + 1
    })

print(f"Created {len(context_docs)} context documents:")
for doc in context_docs:
    print(f"\nRank {doc['rank']}: {doc['metadata']}")
    print(f"Content preview: {doc['content'][:150]}...")

Created 3 context documents:

Rank 1: {'source': '..\\data\\textfiles\\sample1.txt'}
Content preview: Sample Document 1

This is a sample text file for testing.
It contains a few lines of plain text.
You can use it for RAG ingestion experiments.
...

Rank 2: {'source': '..\\data\\textfiles\\sample1.txt'}
Content preview: Sample Document 1

This is a sample text file for testing.
It contains a few lines of plain text.
You can use it for RAG ingestion experiments.
...

Rank 3: {'source': '..\\data\\textfiles\\sample2.txt'}
Content preview: Sample Document 2


Deep Agents overview



The easiest way to start building agents and applications powered by LLMs—with built-in capabilities for t...
